# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [156]:
%load_ext dotenv
%dotenv 

import os
import sys
sys.path.append(os.getenv('SRC_DIR'))

import pandas as pd
import numpy as np

# Set the environment variable
os.environ["FIRES_DATA"] = r"C:\Users\User\dsi\production\05_src\data\fires\forestfires.csv"

ft_file = os.getenv("FIRES_DATA")
df_raw = pd.read_csv(ft_file)

print(df_raw.head())

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
   X  Y month  day  FFMC   DMC     DC  ISI  temp  RH  wind  rain  area
0  7  5   mar  fri  86.2  26.2   94.3  5.1   8.2  51   6.7   0.0   0.0
1  7  4   oct  tue  90.6  35.4  669.1  6.7  18.0  33   0.9   0.0   0.0
2  7  4   oct  sat  90.6  43.7  686.9  6.7  14.6  33   1.3   0.0   0.0
3  8  6   mar  fri  91.7  33.3   77.5  9.0   8.3  97   4.0   0.2   0.0
4  8  6   mar  sun  89.3  51.3  102.2  9.6  11.4  99   1.8   0.0   0.0


In [157]:
# create dataframe from csv file
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [158]:
# define X features from fires_dt dataframe

X_features_df = fires_dt.drop(columns='area')
X_features_df

,coord_x,coord_y,month,day,ffmc,dmc,dc,isi,temp,rh,wind,rain
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
512,4,3,aug,sun,81.6,56.7,665.6,1.9,27.8,32,2.7,0.0
513,2,4,aug,sun,81.6,56.7,665.6,1.9,21.9,71,5.8,0.0
514,7,4,aug,sun,81.6,56.7,665.6,1.9,21.2,70,6.7,0.0
515,1,4,aug,sat,94.4,146.0,614.7,11.3,25.6,42,4.0,0.0


In [159]:
# define Y target from fires_dt dataframe

Y_target_data = np.log1p(fires_dt[['area']])
Y_target_data

,area
0,0.000000
1,0.000000
2,0.000000
3,0.000000
4,0.000000
...,...
512,2.006871
513,4.012592
514,2.498152
515,0.000000


# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [160]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder

# scale numeric variable and one hot encode categorical variables

preproc1 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain'] ),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['month', 'day']), 
    ], remainder='drop'
)

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [161]:
# scale numeric variable and one hot encode categorical variables

preproc2 = ColumnTransformer(
    transformers=[
    ('num', numeric_pipeline, ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['month', 'day'])
   ], remainder='drop'
)

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [162]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate

# choose baseline regressor

baseline_preproc1 = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', LinearRegression())
])

# evaluate using cross-validation

scoring = ['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error']
baseline_1 = cross_validate(baseline_preproc1, X_features_df, Y_target_data, cv=5, scoring = scoring, return_train_score = True)

pd.DataFrame(baseline_1)

,fit_time,score_time,test_r2,train_r2,test_neg_mean_squared_error,train_neg_mean_squared_error,test_neg_mean_absolute_error,train_neg_mean_absolute_error
0,0.040492,0.018766,0.000000,0.114657,-2.652871,-1.818703,-1.543595,-1.111138
1,0.018320,0.013712,-0.171031,0.087048,-1.060721,-2.021173,-0.906740,-1.148103
2,0.009835,0.007861,-0.583156,0.105527,-4.808909,-1.333338,-1.815244,-0.923885
3,0.009891,0.006681,-0.073334,0.082543,-1.816321,-1.836460,-1.111401,-1.087758
4,0.007988,0.006971,-0.751386,0.116232,-3.861594,-1.659828,-1.380462,-1.054097


In [163]:
# choose baseline regressor

baseline_preproc2 = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', LinearRegression())
])

# evaluate using cross-validation

scoring = ['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error']
baseline_2 = cross_validate(baseline_preproc2, X_features_df, Y_target_data, cv=5, scoring = scoring, return_train_score = True)

pd.DataFrame(baseline_2)

,fit_time,score_time,test_r2,train_r2,test_neg_mean_squared_error,train_neg_mean_squared_error,test_neg_mean_absolute_error,train_neg_mean_absolute_error
0,0.059266,0.010255,0.000000,0.135161,-2.862228,-1.776582,-1.596725,-1.098798
1,0.038441,0.010572,-0.134916,0.096617,-1.028008,-1.999988,-0.892306,-1.144266
2,0.029178,0.007408,-0.648403,0.116537,-5.007097,-1.316925,-1.833934,-0.914931
3,0.028433,0.007891,-0.093928,0.095382,-1.851171,-1.810760,-1.118670,-1.079089
4,0.032835,0.007604,-0.237612,0.119850,-2.728785,-1.653034,-1.274666,-1.050988


In [164]:
from sklearn.ensemble import RandomForestRegressor

# choose baseline regressor (advanced)

rf_preproc1 = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', RandomForestRegressor(random_state=99))
])

# evaluate using cross-validation

scoring = ['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error']
rf_1 = cross_validate(baseline_preproc2, X_features_df, Y_target_data, cv=5, scoring = scoring, return_train_score = True)

pd.DataFrame(rf_1)

,fit_time,score_time,test_r2,train_r2,test_neg_mean_squared_error,train_neg_mean_squared_error,test_neg_mean_absolute_error,train_neg_mean_absolute_error
0,0.101136,0.009022,0.000000,0.135161,-2.862228,-1.776582,-1.596725,-1.098798
1,0.035891,0.007213,-0.134916,0.096617,-1.028008,-1.999988,-0.892306,-1.144266
2,0.028899,0.006172,-0.648403,0.116537,-5.007097,-1.316925,-1.833934,-0.914931
3,0.033014,0.005969,-0.093928,0.095382,-1.851171,-1.810760,-1.118670,-1.079089
4,0.032911,0.006437,-0.237612,0.119850,-2.728785,-1.653034,-1.274666,-1.050988


In [165]:

# choose baseline regressor (advanced)

rf_preproc2 = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', RandomForestRegressor(random_state=99))
])  

# evaluate using cross-validation

scoring = ['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error']
rf_2 = cross_validate(baseline_preproc2, X_features_df, Y_target_data, cv=5, scoring = scoring, return_train_score = True)

pd.DataFrame(rf_2)

,fit_time,score_time,test_r2,train_r2,test_neg_mean_squared_error,train_neg_mean_squared_error,test_neg_mean_absolute_error,train_neg_mean_absolute_error
0,0.036880,0.007313,0.000000,0.135161,-2.862228,-1.776582,-1.596725,-1.098798
1,0.035450,0.007677,-0.134916,0.096617,-1.028008,-1.999988,-0.892306,-1.144266
2,0.030769,0.007470,-0.648403,0.116537,-5.007097,-1.316925,-1.833934,-0.914931
3,0.031478,0.007442,-0.093928,0.095382,-1.851171,-1.810760,-1.118670,-1.079089
4,0.027828,0.007897,-0.237612,0.119850,-2.728785,-1.653034,-1.274666,-1.050988


# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [166]:
from sklearn.model_selection import GridSearchCV

# grid search piepline 1

param_grid = {
    'regressor__fit_intercept': [True, False]
}

grid_search_1 = GridSearchCV(
    estimator=baseline_preproc1,
    param_grid=param_grid,
    cv=5,                   
    scoring='r2',           
    return_train_score=True
)

grid_search_1.fit(X_features_df, Y_target_data)

,estimator,Pipeline(step...egression())])
,param_grid,"{'regressor__fit_intercept': [True, False]}"
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,transformers,"[('num', ...), ('cat', ...)]"


In [167]:
print("Pipeline 1 Best Parameters:", grid_search_1.best_params_)
print("Pipeline 1 Best Cross-Validation R² Score:", grid_search_1.best_score_)

Pipeline 1 Best Parameters: {'regressor__fit_intercept': True}
Pipeline 1 Best Cross-Validation R² Score: -0.3157814323147017


In [168]:
# grid search piepline 2

param_grid = {
    'regressor__fit_intercept': [True, False]
}

grid_search_2 = GridSearchCV(
    estimator=baseline_preproc2,
    param_grid=param_grid,
    cv=5,                   
    scoring='r2',           
    return_train_score=True
)

grid_search_2.fit(X_features_df, Y_target_data)

,estimator,Pipeline(step...egression())])
,param_grid,"{'regressor__fit_intercept': [True, False]}"
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,transformers,"[('num', ...), ('cat', ...)]"


In [169]:
print("Pipeline 2 Best Parameters:", grid_search_2.best_params_)
print("Pipeline 2 Best Cross-Validation R² Score:", grid_search_2.best_score_)

Pipeline 2 Best Parameters: {'regressor__fit_intercept': True}
Pipeline 2 Best Cross-Validation R² Score: -0.22297166579285926


In [170]:
# grid search piepline 3

param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [None, 10]
}

grid_search_3 = GridSearchCV(
    estimator=rf_preproc1,
    param_grid=param_grid,
    scoring='r2',     
    cv=5,
    n_jobs=-1,
    
)

grid_search_3.fit(X_features_df, Y_target_data)

c:\Users\User\miniconda3\envs\dsi_production_only\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,estimator,Pipeline(step...m_state=99))])
,param_grid,"{'regressor__max_depth': [None, 10], 'regressor__n_estimators': [100, 200]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [171]:
print("Pipeline 3 Best parameters:", grid_search_3.best_params_)
print("Pipeline 3 Best cross-validated R² score:", grid_search_3.best_score_)

Pipeline 3 Best parameters: {'regressor__max_depth': 10, 'regressor__n_estimators': 100}
Pipeline 3 Best cross-validated R² score: -0.28929254105367186


In [172]:
# grid search piepline 4

param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [None, 10]
}

grid_search_4 = GridSearchCV(
    estimator=rf_preproc2,
    param_grid=param_grid,
    scoring='r2',     # or use 'neg_mean_squared_error'
    cv=5,
    n_jobs=-1,
    
)

grid_search_4.fit(X_features_df, Y_target_data)

c:\Users\User\miniconda3\envs\dsi_production_only\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,estimator,Pipeline(step...m_state=99))])
,param_grid,"{'regressor__max_depth': [None, 10], 'regressor__n_estimators': [100, 200]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [173]:
print("Pipeline 4 Best parameters:", grid_search_4.best_params_)
print("Pipeline 4 Best cross-validated R² score:", grid_search_4.best_score_)

Pipeline 4 Best parameters: {'regressor__max_depth': 10, 'regressor__n_estimators': 100}
Pipeline 4 Best cross-validated R² score: -0.2898905688563308


# Evaluate

+ Which model has the best performance?

In [174]:
import pandas as pd

# compare best scores across all four pipelines

model_comparison = pd.DataFrame({
    'Model': ['Pipeline 1: LinearRegression', 'Pipeline 2: LinearRegression','Pipeline 3 : RandomForest', 'Pipeline 4: RandomForest'],
    'Best R2': [
        grid_search_1.best_score_,
        grid_search_2.best_score_,
        grid_search_3.best_score_,
        grid_search_4.best_score_
    ]
})

print(model_comparison)

                          Model   Best R2
0  Pipeline 1: LinearRegression -0.315781
1  Pipeline 2: LinearRegression -0.222972
2     Pipeline 3 : RandomForest -0.289293
3      Pipeline 4: RandomForest -0.289891


The models are all performing worse than the mean, I have no idea why I am getting all negative R2 values - please help!!! I am assuming it has something to do with the categorical variables, but I cannot figure it out.

Pipeline 2 has the least worst performing model of all four.

# Export

+ Save the best performing model to a pickle file.

In [175]:
import pickle

In [176]:
with open('best_model.pkl', 'wb') as f:
    pickle.dump(grid_search_2.best_estimator_, f)

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.